## Data Ingestion

In [ ]:
import pandas as pd
path_oct = "../data/raw/2019-Oct.csv"
path_nov = "../data/raw/2019-Nov.csv"
chunksize = 10 ** 6
chunks_oct = []
cols = ["event_time", "event_type", "user_id", "price"]
parse_dates = ["event_time"]
with pd.read_csv(path_oct, chunksize=chunksize, usecols=cols, parse_dates=parse_dates, dtype={'event_type' : 'category'}) as reader:
    for chunk in reader:
        chunks_oct.append(chunk)
        
df_oct = pd.concat(chunks_oct, ignore_index=True)
print(df_oct.head())

chunks_nov = []

with pd.read_csv(path_nov, chunksize=chunksize, usecols=cols, parse_dates=parse_dates, dtype={'event_type' : 'category'}) as reader:
    for chunk in reader:
        chunks_nov.append(chunk)
df_nov = pd.concat(chunks_nov, ignore_index=True)
print(df_nov.head())

                 event_time event_type    price    user_id
0 2019-10-01 00:00:00+00:00       view    35.79  541312140
1 2019-10-01 00:00:00+00:00       view    33.20  554748717
2 2019-10-01 00:00:01+00:00       view   543.10  519107250
3 2019-10-01 00:00:01+00:00       view   251.74  550050854
4 2019-10-01 00:00:04+00:00       view  1081.98  535871217


In [2]:
df_total = pd.concat([df_oct, df_nov], ignore_index=True)

In [3]:
df_total.tail()

,event_time,event_type,price,user_id
109950738,2019-11-30 23:59:58+00:00,view,277.74,532714000
109950739,2019-11-30 23:59:58+00:00,view,62.81,545223467
109950740,2019-11-30 23:59:59+00:00,view,167.03,557794415
109950741,2019-11-30 23:59:59+00:00,view,566.27,531607492
109950742,2019-11-30 23:59:59+00:00,view,1312.52,579969851


## Data Processing and Feature Engineering

In [4]:
df_total["week"] = df_total["event_time"].dt.isocalendar().week
df_total["weekday"] = df_total["event_time"].dt.dayofweek
print(df_total.head())

                 event_time event_type    price    user_id  week  weekday
0 2019-10-01 00:00:00+00:00       view    35.79  541312140    40        1
1 2019-10-01 00:00:00+00:00       view    33.20  554748717    40        1
2 2019-10-01 00:00:01+00:00       view   543.10  519107250    40        1
3 2019-10-01 00:00:01+00:00       view   251.74  550050854    40        1
4 2019-10-01 00:00:04+00:00       view  1081.98  535871217    40        1


In [5]:
df_total["price"].isna().sum()

np.int64(0)

In [6]:
(df_total["price"] <= 0).sum()

np.int64(256761)

In [7]:
df_total = df_total[df_total["price"] > 0]
df_total.shape

(109693982, 6)

In [8]:
df_total["price_range"] = pd.qcut(q=3, labels=["Low", "Medium", "High"], x=df_total["price"])
df_total.head()

,event_time,event_type,price,user_id,week,weekday,price_range
0,2019-10-01 00:00:00+00:00,view,35.79,541312140,40,1,Low
1,2019-10-01 00:00:00+00:00,view,33.20,554748717,40,1,Low
2,2019-10-01 00:00:01+00:00,view,543.10,519107250,40,1,High
3,2019-10-01 00:00:01+00:00,view,251.74,550050854,40,1,Medium
4,2019-10-01 00:00:04+00:00,view,1081.98,535871217,40,1,High


In [9]:
df_total["price_range"].value_counts()

price_range
Medium    36571609
Low       36565711
High      36556662
Name: count, dtype: int64

## Funnel Analysis

In [10]:
df_view = df_total[df_total["event_type"] == "view"]
unique_users_view = df_view["user_id"].nunique()
print(f"Number of unique users that viewed a product: {unique_users_view}")

df_cart = df_total[df_total["event_type"] == "cart"]
unique_users_cart = df_cart["user_id"].nunique()
print(f"Number of unique users that placed products after viewing in their cart: {unique_users_cart}")

df_purchase = df_total[df_total["event_type"] == "purchase"]
unique_users_purchase= df_purchase["user_id"].nunique()
print(f"Number of unique users that purchased a product in their carts: {unique_users_purchase}")

Number of unique users that viewed a product: 5314747
Number of unique users that placed products after viewing in their cart: 1053550
Number of unique users that purchased a product in their carts: 697470


In [11]:
cart_conversion_rate = (unique_users_cart / unique_users_view) * 100
print(f"Percentage of unique users that placed a product in their cart after viewing it: {round(cart_conversion_rate, 2)}%")

purchase_conversion_rate = (unique_users_purchase / unique_users_cart) * 100
print(f"Percentage of unique users that purchased a product after placing it in their cart: {round(purchase_conversion_rate, 2)}%")


Percentage of unique users that placed a product in their cart after viewing it: 19.82%
Percentage of unique users that purchased a product after placing it in their cart: 66.2%


In [25]:
import plotly.express as px
funnel_data = dict(
    number = [unique_users_view, unique_users_cart, unique_users_purchase],
    event = ["View", "Cart", "Purchase"]
)
df_funnel = pd.DataFrame(funnel_data)
df_funnel["prct"] = [100, round(cart_conversion_rate, 2), round(purchase_conversion_rate, 2)]
fig = px.funnel(df_funnel, x = "number", y = "event", title="E-Commerce Funnel Analysis", text="prct")
fig.update_traces(textposition='inside',texttemplate="%{x:.0f\n}<br>%{text:.2f}"+'%')
fig.show()

In [40]:
df_funnel_price = df_total.groupby(["event_type", "price_range"])["user_id"].nunique().reset_index()
df_funnel_price = df_funnel_price.rename(columns={"user_id": "unique_users"})
df_funnel_price

,event_type,price_range,unique_users
0,cart,Low,389474
1,cart,Medium,506958
2,cart,High,437081
3,purchase,Low,267970
4,purchase,Medium,333056
5,purchase,High,261118
6,view,Low,3026585
7,view,Medium,3292068
8,view,High,3190543


In [ ]:
fig = px.funnel(df_funnel_price, x = "unique_users", y = "event_type", color = "price_range", category_orders={"event_type": ["purchase", "cart", "view"]},)
fig.show()